# Task 2 — Season Classification

**Status:** execution-ready final scaffold. The protocol and notebook story are fixed, but no model result is claimed before real execution.

**How to finish this notebook:** implement reusable logic under `src/fashion/`; fill each code cell; run it; keep its output; then replace the prompt text after the bold Interpretation marker with a short evidence-based finding.

**Structural rule:** every leaf `###` subsection owns exactly one code cell. A broader `###` owns no direct code cell; divide it into `####` subsubsections, with one code cell per `####` leaf.

**Frozen boundaries:** teacher images only; `data/processed/splits.csv` is the only split; all submitted models are trained from scratch; all runs go through `fashion.train.registry` to `results/runs.csv`; holdout remains sealed until Notebook 06.

**Core risks:** weak-visual-signal labels, article-type shortcut learning, class imbalance, acquisition artifacts, and calibration failure.

**Primary development metric:** pooled out-of-fold macro-F1 over `Fall`, `Spring`, `Summer`, and `Winter`.

## 1. Task contract and reproducibility

Freeze the decision, labels, paths, seed, and execution environment before any result is viewed.

### 1.1 Frozen task configuration

This cell creates the single configuration object used by every later cell.

In [ ]:
# TODO:
# - Import only orchestration helpers; keep reusable implementation in src/fashion/.
# - Declare target=season, labels=[Fall, Spring, Summer, Winter], and seed=2753.
# - Record the saved CV digest.
# - Declare Task 2 evidence, figure, checkpoint, and final-model paths.
# - Create the result directories and display a one-row configuration table.
#
# Expected output:
# - A configuration table containing the target, labels, seed, split digest, and artifact paths.
# - No holdout path, label, or evaluation loader is opened.

> **Interpretation to write after running:** Confirm that the output matches the assignment contract. Explain that inference is image-only and that the official CSV must always contain one Season label.

### 1.2 Environment and deterministic execution

Package versions, hardware, and random state must be recorded so runs can be reproduced.

In [ ]:
# TODO:
# - Call the shared reproducibility helper with seed 2753.
# - Collect Python, package, CUDA, GPU/CPU, deterministic-setting, and mixed-precision information.
# - Write the environment record to results/evidence/task2/environment.json and display it.
#
# Expected output:
# - A compact environment table and a saved JSON artifact.
# - The same seed and deterministic policy that will be stored in every registry row.

> **Interpretation to write after running:** State the actual machine and software used. Note any deterministic-operation limitation rather than hiding it.

## 2. Data and EDA handoff

Reproduce only the Task 2 facts needed from Notebook 01. These are data facts, not model results.

### 2.1 Load the development task frame

Task 2 uses teacher images and excludes only rows with a truly blank Season label.

In [ ]:
# TODO:
# - Load the canonical redacted split through fashion.data.dataset.load_splits().
# - Select development rows and apply has_season_label.
# - Join only the safe fields needed for training and later slicing.
# - Assert 32,773 development rows, 32,753 valid Season rows, and 20 blank Season labels.
# - Display valid, missing, quarantine-visible, and holdout-visible counts.
#
# Expected output:
# - 32,753 valid rows and 20 masked rows.
# - Zero visible holdout labels and zero quarantine rows in the modelling frame.

> **Interpretation to write after running:** Confirm the counts against Notebook 01. If any value differs, stop and explain the mismatch before modelling.

### 2.2 Audit class balance, folds, and EDA evidence

This topic needs separate numeric and visual evidence, so it is divided into two subsubsections.

#### 2.2.1 Reproduce class and fold statistics

The numeric audit checks the exact EDA facts that drive imbalance and validation choices.

In [ ]:
# TODO:
# - Calculate Season counts, percentages, largest-to-smallest ratio, and per-fold support.
# - Assert Summer=16,235, Fall=8,928, Winter=6,261, Spring=1,329 and all four labels in every fold.
# - Save results/evidence/task2/eda_handoff.csv.
#
# Expected output:
# - Exact class counts, a 12.22:1 ratio, and a five-fold support table.
# - A traceable CSV generated from development rows only.

> **Interpretation to write after running:** Explain why accuracy favours Summer and why Spring needs equal weight in evaluation.

#### 2.2.2 Display the saved shortcut and transform evidence

The visual handoff keeps Notebook 03 tied to the acquisition, compression, shortcut, and transform risks found in Notebook 01.

In [ ]:
# TODO:
# - Load and display the saved acquisition, file-size, shortcut-heatmap, and transform-risk figures.
# - Verify that every file exists and record its path and digest.
# - Do not recompute protected data or reinterpret descriptive lookups as model accuracy.
#
# Expected output:
# - Four cited EDA figures with valid paths and digests.
# - No model-performance claim.

> **Interpretation to write after running:** State the hypotheses suggested by each figure. Keep year and file size as diagnostic slices, not inputs.

## 3. Development-validation protocol

Use all five canonical folds and create one out-of-fold prediction per valid product.

### 3.1 Build the five canonical fold views

Every candidate must see exactly the same training and validation rows.

In [ ]:
# TODO:
# - Use fashion.data.dataset.iter_cv_folds() to build five training/validation views.
# - Apply has_season_label after obtaining each fold view.
# - Summarise row count, label support, and product_family_group count for train and validation.
# - Assert that no product_family_group crosses the two sides of any round.
#
# Expected output:
# - Exactly five fold summaries.
# - All four labels in every training and validation view.
# - Zero family crossing and no new split.

> **Interpretation to write after running:** State that all-five-fold CV was chosen for stable comparison and that no best-looking fold will be selected.

### 3.2 Freeze the OOF coverage contract

Pooled evidence is valid only when each product is predicted outside its training data exactly once.

In [ ]:
# TODO:
# - Create the expected OOF index from valid development IDs and their saved cv_fold.
# - Assert unique IDs, one prediction per ID, and allowed labels.
# - Reject a missing or incomplete probability vector.
# - Save the empty contract/schema to results/evidence/task2/oof_contract.json.
#
# Expected output:
# - Expected OOF row count of 32,753.
# - A unique product-ID contract with no holdout or quarantine membership.

> **Interpretation to write after running:** Explain why pooled OOF predictions are the fair unit for model comparison.

### 3.3 Freeze metrics and selection rules

Metrics must be declared before candidate results are visible.

In [ ]:
# TODO:
# - Configure pooled OOF macro-F1 with the four labels in fixed order and zero_division=0.
# - Configure fold mean/SD, per-class metrics, balanced accuracy, NLL, and Brier.
# - Configure reliability outputs.
# - Record the tie rule: below a 0.5-point macro-F1 difference and an interval
#   containing zero, prefer the smaller model when robustness is within one point.
# - Save the metric card to results/evidence/task2/metric_contract.json.
#
# Expected output:
# - A versioned metric and tie-break card written before model execution.
# - No accuracy-only winner rule.

> **Interpretation to write after running:** Explain how macro-F1 gives Spring the same voting weight as Summer and how secondary metrics expose different failure types.

## 4. Preprocessing and leakage controls

Fit every learned image value inside the current training folds and keep diagnostic metadata out of the model input.

### 4.1 Fit and inspect the fold-specific image pipeline

Aspect ratio and training-only statistics protect the small 60 x 80 images from distortion and leakage.

In [ ]:
# TODO:
# - Build EXIF transpose, RGB conversion, and aspect-preserving resize.
# - Add neutral padding and tensor conversion.
# - Fit mean and standard deviation on content pixels from the current training folds only.
# - Apply the frozen values to validation without refitting.
# - Display one batch, its shape/range, content mask, class labels, and transform statistics.
#
# Expected output:
# - Undistorted product examples with a valid image batch.
# - Fold-specific training statistics and no validation-fitted value.

> **Interpretation to write after running:** Confirm that product shape is preserved. Explain any unusual colour or padding behaviour visible in the batch.

### 4.2 Declare the controlled transform comparison

P0/P1 and A0/A1 answer whether upscaling and mild colour change help without changing the model budget.

In [ ]:
# TODO:
# - Create P0=(80,60), P1=(128,96), and A0=mild geometry transform IDs.
# - Define A1 as A0 plus mild colour jitter.
# - Build a four-row matrix with one ResNet18 config, seed, folds, and budget.
# - Predeclare the rejection rule.
# - Write the matrix to results/evidence/task2/transform_matrix.csv.
#
# Expected output:
# - Four traceable transform configurations.
# - All non-transform variables held equal.

> **Interpretation to write after running:** Explain which visual question each comparison answers. Do not select a transform until the controlled results exist.

### 4.3 Run the leakage and transform audit

Year, file size, and true ArticleType can define later slices but must never enter the inference tensor.

In [ ]:
# TODO:
# - Inspect dataset outputs and model-call signatures.
# - Assert that model features contain image tensors only.
# - Assert that validation transforms are deterministic.
# - Assert that year, compressed file size, and true ArticleType are absent from inference inputs.
# - Write the pass/fail checklist to results/evidence/task2/leakage_audit.json.
#
# Expected output:
# - All leakage checks pass.
# - A traceable audit proving image-only inference and validation without refitting.

> **Interpretation to write after running:** State exactly which metadata is retained only for post-prediction analysis and why that does not leak into training.

## 5. Baselines

Establish simple comparison anchors before interpreting deep models.

### 5.1 B0 training-fold majority baseline

B0 verifies fold handling and shows what the 49.57% Summer majority can achieve.

In [ ]:
# TODO:
# - Fit the majority Season label separately inside each training fold.
# - Predict that label for the matching validation fold and append a registry row.
# - Combine the five validation outputs into B0 OOF predictions.
# - Save predictions and metrics under results/evidence/task2/b0_majority/.
#
# Expected output:
# - 32,753 unique B0 OOF predictions.
# - A valid registry entry for every fold and a pooled metric table.

> **Interpretation to write after running:** Compare the result with the class distribution. Explain why this baseline can have reasonable accuracy but poor macro-F1 and Spring recall.

### 5.2 B1 HOG plus HSV linear-SVM baseline

B1 tests whether hand-designed shape and colour features already carry useful Season signal.

In [ ]:
# TODO:
# - Extract HOG and HSV-histogram features from the fold-fitted image pipeline.
# - Fit feature scaling and the linear SVM on each training fold only.
# - Generate five-fold OOF labels and decision scores, register every run, and save artifacts.
# - Report primary and per-class metrics; do not invent probabilities for calibration metrics.
#
# Expected output:
# - 32,753 unique B1 OOF predictions and decision scores.
# - A fair comparison with B0 using the same fold membership.

> **Interpretation to write after running:** State whether explicit shape and colour improve balanced class performance, especially Spring, and identify the main remaining confusions.

## 6. Scratch model families

Define three different deep-learning capacities and prove that the submitted candidates start from random weights.

### 6.1 Define the scratch model factories

Each architecture is a separate implementation unit and therefore receives its own subsubsection and code cell.

#### 6.1.1 C1 four-block SmallCNN

C1 is the simple deep baseline and makes capacity and failure analysis easy to explain.

In [ ]:
# TODO:
# - Implement the C1 factory in fashion.models.season with four convolutional blocks.
# - Use random Kaiming initialisation and return four Season logits.
# - Expose the same model-factory interface used by all later experiments.
#
# Expected output:
# - A model summary with output dimension four.
# - No external checkpoint or pretrained weight.

> **Interpretation to write after running:** Explain why C1 is a useful capacity baseline and what it may miss on small catalogue images.

#### 6.1.2 C2 ResNet18 with a small-image stem

C2 tests residual learning without discarding small-image detail in the first layers.

In [ ]:
# TODO:
# - Create ResNet18 with weights=None.
# - Replace the first layer with a 3 x 3 stride-1 convolution and remove the first max-pool.
# - Replace the classifier with four Season logits and expose the shared factory interface.
#
# Expected output:
# - A modified ResNet18 summary with four outputs.
# - Proof of weights=None and the small-image stem.

> **Interpretation to write after running:** Explain why the stem differs from the standard ImageNet design and how residual capacity may help.

#### 6.1.3 C3 MobileNetV3-Small

C3 tests whether a mobile architecture offers a better quality-to-latency trade-off.

In [ ]:
# TODO:
# - Create MobileNetV3-Small with weights=None.
# - Replace its classifier with four Season logits.
# - Expose the shared factory interface and retain mobile-friendly operations.
#
# Expected output:
# - A MobileNetV3-Small summary with four outputs.
# - Proof of random initialisation and no downloaded weights.

> **Interpretation to write after running:** Explain the expected parameter and latency advantage and why it must be measured rather than assumed.

### 6.2 Audit shapes, parameters, and random initialisation

A forward-pass and weight-provenance audit catches architecture mistakes before GPU runs.

In [ ]:
# TODO:
# - Run one synthetic and one real batch through every model.
# - Assert logits shape [batch,4], finite values, and compatible input size.
# - Record parameter count, trainable parameter count, initial-weight hash, and scratch=true.
# - Save the audit to results/evidence/task2/scratch_model_audit.csv.
#
# Expected output:
# - Three successful forward passes.
# - A scratch-compliance and capacity table linked to model IDs.

> **Interpretation to write after running:** Compare capacity and expected deployment cost. Confirm that each submitted candidate is scratch-trained.

### 6.3 Declare the pretrained benchmark boundary

A pretrained comparison is useful only when it is clearly excluded from final eligibility.

In [ ]:
# TODO:
# - Configure P* as a separate ResNet benchmark with benchmark_only=true and scratch=false.
# - Keep its folds, input size, budget, and metric comparable where possible.
# - Add an eligibility field that prevents P* from becoming the submitted winner.
# - Display and save the benchmark-boundary record before running it.
#
# Expected output:
# - A visible benchmark-only warning and final_eligible=false.
# - No ambiguity between the comparison benchmark and submitted scratch models.

> **Interpretation to write after running:** Explain what the pretrained gap says about data and representation learning, without selecting P* as the final model.

## 7. Training and run registry

Prove the shared engine, checkpointing, failure handling, and registry before long runs.

### 7.1 G0 tiny-batch smoke test

A tiny overfit test catches label order, image mapping, loss, and backpropagation errors cheaply.

In [ ]:
# TODO:
# - Select 256-512 training images from fold 0 without touching validation labels for tuning.
# - Run two smoke epochs and a tiny-batch overfit check through the shared engine.
# - Verify decreasing loss, finite gradients, and expected labels.
# - Verify that the checkpoint state can be restored.
# - Register the smoke run with stage=smoke and exclude it from comparison tables.
#
# Expected output:
# - A clear loss decrease or a diagnosed failure.
# - One restorable checkpoint and one traceable smoke registry row.

> **Interpretation to write after running:** Explain whether the pipeline learned the tiny sample. If it did not, diagnose the cause before any full run.

### 7.2 Audit registry and checkpoint traceability

Every reported number must be traceable to configuration, split, code, and artifact hashes.

In [ ]:
# TODO:
# - Load results/runs.csv through fashion.train.registry.
# - Validate required fields, unique run IDs, split/config hashes, and status.
# - Validate hardware and checkpoint paths.
# - Append and verify one deliberate failed-run record without hiding its error.
# - Display a compact registry health table.
#
# Expected output:
# - Valid success and failure rows with stable schema.
# - No hand-written final metric or orphaned checkpoint.

> **Interpretation to write after running:** Confirm that each future table can be regenerated from run IDs. Explain any failed run honestly.

## 8. Controlled experiment matrix

Execute broad comparisons under equal budgets, then spend extra compute only on justified finalists.

### 8.1 G1 and G3 model-family comparison

Screening and full finalist training answer different questions, so they are separated below.

#### 8.1.1 G1 equal-budget family screening

The screen decides which families deserve more compute under one fixed budget.

In [ ]:
# TODO:
# - Run B0, B1, C1, C2, and C3 over all five folds.
# - Use the declared eight-epoch screen where applicable.
# - Keep transforms, seed, effective batch size, folds, and metric fixed.
# - Register every success and failure and save complete OOF artifacts.
#
# Expected output:
# - Complete five-fold screening rows for every candidate.
# - One OOF artifact per candidate covering all valid IDs.

> **Interpretation to write after running:** Compare families broadly and name the two scratch deep finalists without using only the best fold.

#### 8.1.2 G3 full-budget finalist comparison

The finalist run tests the strongest families with equal training opportunity.

In [ ]:
# TODO:
# - Train the two selected scratch families with the same full budget and early-stopping rule.
# - Use all five folds and preserve the frozen preprocessing and metric contract.
# - Save learning histories, checkpoints, registry rows, and OOF predictions.
#
# Expected output:
# - Two complete full-budget five-fold candidate records.
# - Stable, traceable learning histories and OOF evidence.

> **Interpretation to write after running:** Explain whether the full budget changes the screening conclusion and why each other family remains rejected.

### 8.2 G2 transform and compact tuning ablations

Input size, augmentation, and tuning are separate factors. Each receives its own controlled subsubsection.

#### 8.2.1 P0 versus P1 input size

This ablation tests whether moderate upscaling helps the small source images.

In [ ]:
# TODO:
# - Run P0=(80,60) versus P1=(128,96) on the fixed C2 screen configuration.
# - Hold augmentation, seed, folds, budget, and optimisation settings constant.
# - Save the paired fold and pooled comparison.
#
# Expected output:
# - One controlled input-size table with OOF metrics and cost.
# - No simultaneous augmentation change.

> **Interpretation to write after running:** State whether upscaling adds useful signal or only compute and interpolation.

#### 8.2.2 A0 versus A1 augmentation

This ablation tests whether mild colour jitter helps generalisation without erasing genuine seasonal colour cues.

In [ ]:
# TODO:
# - Run A0=mild geometry versus A1=A0 plus mild colour jitter using the selected input size.
# - Hold model, seed, folds, budget, and optimisation settings constant.
# - Save overall, per-class, and robustness differences.
#
# Expected output:
# - One controlled augmentation table.
# - Spring and colour-sensitive class evidence.

> **Interpretation to write after running:** Explain whether colour jitter improves robustness or removes useful Season information.

#### 8.2.3 Compact finalist tuning

A small predeclared search avoids an open-ended hunt for a lucky configuration.

In [ ]:
# TODO:
# - Run exactly the three frozen learning-rate and weight-decay pairs on finalists.
# - Keep all other variables fixed and use the same five folds.
# - Select with the frozen primary metric and tie rule.
#
# Expected output:
# - A three-row tuning table per finalist.
# - One justified configuration without post-result search expansion.

> **Interpretation to write after running:** Explain the sensitivity to tuning and whether the gain is large enough to matter.

### 8.3 G4 problem-driven improvements and benchmark

Class balance, multi-task learning, and pretrained comparison test different hypotheses and are separated below.

#### 8.3.1 I1 class-balanced loss

I1 directly tests whether fold-fitted reweighting improves the Spring minority.

In [ ]:
# TODO:
# - Fit class-balanced weights independently inside every training fold.
# - Run the selected scratch architecture over all five folds with all other settings fixed.
# - Save overall and per-class OOF evidence.
#
# Expected output:
# - Complete I1 OOF predictions and registry rows.
# - Spring recall/F1 change plus majority-class trade-offs.

> **Interpretation to write after running:** State whether I1 solves minority neglect and whether it harms other classes or calibration.

#### 8.3.2 I2 image-only multi-task training

I2 tests whether auxiliary ArticleType supervision improves shared visual features or causes negative transfer.

In [ ]:
# TODO:
# - Train Season as the main target and ArticleType as the auxiliary target for lambda 0.1 and 0.3.
# - Keep inference image-only and never pass true ArticleType at prediction time.
# - Measure overall, Spring, and ArticleType aligned/conflict performance.
#
# Expected output:
# - Two complete lambda comparisons.
# - Negative-transfer and shortcut-conflict evidence.

> **Interpretation to write after running:** Explain whether auxiliary supervision helps genuine Season learning or strengthens the ArticleType shortcut.

#### 8.3.3 P* pretrained benchmark execution

P* estimates the value of transferred representation while remaining ineligible for submission.

In [ ]:
# TODO:
# - Run the predeclared pretrained ResNet benchmark with benchmark_only=true and scratch=false.
# - Match folds, input, budget, and metric as closely as the comparison permits.
# - Keep it outside the eligible winner table.
#
# Expected output:
# - A traceable pretrained comparison result.
# - Visible final_eligible=false in every artifact.

> **Interpretation to write after running:** Explain the representation-learning gap without treating P* as a permitted final model.

### 8.4 G5 second-seed stability

A second seed checks whether the final ordering depends on random initialisation.

In [ ]:
# TODO:
# - Run the final two eligible candidates with a second frozen seed over all five folds.
# - Compare candidate ordering, per-fold spread, Spring recall, and runtime across seeds.
# - Save the stability table to results/evidence/task2/seed_stability.csv.
#
# Expected output:
# - Ten additional finalist runs and one seed-stability table.
# - A documented stable result or an honest reversal.

> **Interpretation to write after running:** State whether the winner is stable. If ordering reverses, reduce the strength of the final claim.

## 9. Cross-validated results

Turn registry-backed OOF artifacts into the main comparison evidence.

### 9.1 OOF leaderboard and fold stability

The leaderboard must be generated from artifacts, not typed by hand.

In [ ]:
# TODO:
# - Load complete eligible runs and OOF predictions from the registry.
# - Verify 32,753 unique predictions for each compared configuration.
# - Calculate pooled OOF macro-F1, accuracy, and balanced accuracy.
# - Calculate fold mean/SD, runtime, and parameter count.
# - Save and display results/evidence/task2/model_leaderboard.csv.
#
# Expected output:
# - One sorted scorecard covering quality, stability, and cost.
# - No incomplete run or benchmark-only model mixed into final eligibility.

> **Interpretation to write after running:** Describe the meaningful differences, not just the top row. Mention spread, cost, and any near-tie.

### 9.2 Per-class performance and confusion matrices

Per-class evidence shows whether overall improvement is real or only comes from Summer.

In [ ]:
# TODO:
# - Calculate precision, recall, F1, and support for all four labels in fixed order.
# - Create count and row-normalised confusion matrices for finalists.
# - Save the table and report-ready figures under the Task 2 evidence directories.
#
# Expected output:
# - A complete four-class metric table.
# - Readable confusion matrices with traceable run IDs.

> **Interpretation to write after running:** Name the largest confusion pairs and explain whether Spring improved or remained the main limitation.

### 9.3 Calibration and learning behaviour

Confidence quality and epoch behaviour require different figures, so they are split below.

#### 9.3.1 Calibration and risk-coverage

Reliable confidence supports the app's human-review decision.

In [ ]:
# TODO:
# - Calculate NLL, multiclass Brier score, and reliability bins.
# - Calculate risk-coverage from finalist OOF probabilities.
# - Fit any temperature scalar from development OOF logits only.
# - Save calibration tables and report-ready figures.
#
# Expected output:
# - Reliability and risk-coverage evidence for finalists.
# - No holdout-fitted calibration.

> **Interpretation to write after running:** State whether confidence matches correctness and how a review threshold could be chosen.

#### 9.3.2 Learning curves and epoch behaviour

Train and validation curves explain whether the fixed budget and refit epoch rule are reasonable.

In [ ]:
# TODO:
# - Plot loss and frozen primary metric by epoch and fold for finalists.
# - Mark best epochs selected by the predeclared rule.
# - Calculate the median best epoch used for final refit.
#
# Expected output:
# - Traceable learning curves and a median-best-epoch value.
# - Evidence of underfit, overfit, or stable convergence.

> **Interpretation to write after running:** Explain the learning behaviour and justify the final refit epoch without using holdout.

## 10. Error and shortcut analysis

Test the exact weaknesses identified by EDA using OOF predictions only.

### 10.1 Spring and ArticleType shortcut slices

Minority-class failure and shortcut conflict are distinct analyses, so each has its own code cell.

#### 10.1.1 Spring minority analysis

Spring needs direct evidence because it represents only 4.06% of valid labels.

In [ ]:
# TODO:
# - Report Spring precision, recall, F1, confidence, and main confusion destinations for finalists.
# - Include support and changes from B0/B1 and the unweighted-loss model.
# - Save the Spring evidence table.
#
# Expected output:
# - Spring-specific quality and confidence metrics.
# - Traceable comparison across baseline, finalist, and I1.

> **Interpretation to write after running:** Explain whether the model still ignores Spring and which classes absorb its errors.

#### 10.1.2 ArticleType aligned versus conflict

The conflict slice tests whether image performance survives when the ArticleType shortcut is wrong.

In [ ]:
# TODO:
# - Fit ArticleType-to-Season majority mappings on each training fold only.
# - Split its validation rows into aligned and conflict groups after prediction.
# - Compare support, macro-F1, per-class recall, and confidence for finalists and I2.
#
# Expected output:
# - Aligned/conflict slice metrics with fold-fitted mappings.
# - No true ArticleType passed into the model.

> **Interpretation to write after running:** Explain whether improvements reflect broader visual learning or stronger ArticleType dependence.

### 10.2 Acquisition year and compression slices

Acquisition era and file compression are separate shortcut warnings and are analysed independently.

#### 10.2.1 Acquisition-year slice

The year slice tests the strong collection-era association found in development EDA.

In [ ]:
# TODO:
# - Compare 2011-2012 with all other years using metadata after prediction only.
# - Report support, macro-F1, per-class recall, and confidence.
# - Save the year-slice table with run IDs.
#
# Expected output:
# - Two year-group rows per finalist with support.
# - No year feature in model input.

> **Interpretation to write after running:** Describe any gap as acquisition-shortcut risk, not proof that year causes the prediction.

#### 10.2.2 Training-fitted file-size slice

File-size quartiles test sensitivity to compression traces without using file bytes as a feature.

In [ ]:
# TODO:
# - Fit file-size quartile boundaries inside each training fold.
# - Apply those boundaries to matching validation predictions.
# - Report support, macro-F1, per-class recall, and confidence by quartile.
#
# Expected output:
# - Fold-fitted boundaries and file-size slice metrics.
# - No global-development boundary fitted during CV.

> **Interpretation to write after running:** Explain whether the model is unusually dependent on heavily compressed images.

### 10.3 Family-size and image-mode slices

Related-product structure and rare image mode answer different generalisation questions.

#### 10.3.1 Product-family size slice

This slice checks whether results are stronger for products with related rows.

In [ ]:
# TODO:
# - Compare singleton with multi-row product_family_group members.
# - Report support, macro-F1, per-class recall, and uncertainty for finalists.
# - Keep family definitions exactly as delivered.
#
# Expected output:
# - Family-size metrics with support counts.
# - No claim that conservative groups are verified SKUs.

> **Interpretation to write after running:** State whether quality depends on related-product groups and how that limits generalisation.

#### 10.3.2 Greyscale versus RGB slice

Rare greyscale images may reveal colour dependence or transform problems.

In [ ]:
# TODO:
# - Use the existing structural audit field to separate greyscale and RGB validation images.
# - Report support, macro-F1, per-class recall, and confidence.
# - Attach uncertainty warnings to the small greyscale slice.
#
# Expected output:
# - Image-mode metrics with support.
# - A clear small-sample warning where required.

> **Interpretation to write after running:** Explain whether the result indicates colour reliance, while avoiding strong claims from the rare slice.

## 11. Robustness and efficiency

Measure controlled image degradation and the practical cost of the final candidates.

### 11.1 Deterministic perturbation tests

JPEG, brightness, and blur tests check whether mild real-world changes break the same frozen model.

In [ ]:
# TODO:
# - Evaluate unchanged images and JPEG quality 85 re-encoding.
# - Evaluate brightness -15%, brightness +15%, and blur radius 1.
# - Reuse the frozen model and preprocessing; do not tune on degraded views.
# - Report absolute metrics and drops from the unchanged images.
# - Save the robustness table and one report-ready figure.
#
# Expected output:
# - Deterministic perturbation results for each finalist.
# - A clear robustness drop for every controlled change.

> **Interpretation to write after running:** Explain which perturbation is most harmful and whether it changes the deployment recommendation.

### 11.2 Deployment cost

Inference latency and resource size use different measurement protocols, so they are separated below. Seed stability is already handled in Section 8.4.

#### 11.2.1 Batch-one inference latency

The app serves one uploaded image at a time, so batch-one timing is the relevant cost.

In [ ]:
# TODO:
# - Measure warmed CPU and GPU batch-one latency with p50 and p95 under one fixed protocol.
# - Use the exact frozen preprocessing and repeat enough times.
# - Save hardware, warm-up, repeat count, and timing results.
#
# Expected output:
# - Reproducible CPU/GPU p50 and p95 latency per finalist.
# - No training or data-loading time mixed into inference timing.

> **Interpretation to write after running:** State whether the latency is acceptable for interactive use and which model is fastest.

#### 11.2.2 Model and training resource cost

Model size, RAM/VRAM, and training time complete the practical comparison.

In [ ]:
# TODO:
# - Record parameter count, checkpoint bytes, peak RAM/VRAM, and total training time.
# - Join these values with the finalist scorecard by run ID.
# - Save results/evidence/task2/efficiency.csv.
#
# Expected output:
# - A traceable resource-cost table.
# - Measured values for every finalist.

> **Interpretation to write after running:** Explain whether a quality gain justifies its storage and compute cost.

## 12. Explainability and failure cases

Use deterministic examples to understand where the model looks and why it fails.

### 12.1 Select representative examples deterministically

Fixed selection prevents cherry-picking attractive successes or dramatic failures.

In [ ]:
# TODO:
# - Select three correct and three incorrect OOF examples per class.
# - Order candidates by confidence and then product ID.
# - Include high-confidence errors and low-confidence correct cases where available.
# - Save the selected ID list before viewing Grad-CAM.
#
# Expected output:
# - Up to 24 traceable examples with fixed selection reasons.
# - Balanced coverage of all four classes and both outcomes.

> **Interpretation to write after running:** Explain why the selection rule is fair and note any class with too few qualifying examples.

### 12.2 Generate Grad-CAM diagnostics

Grad-CAM can reveal whether the model attends to the product or to borders and background.

In [ ]:
# TODO:
# - Generate Grad-CAM for the frozen selected IDs and a declared convolutional layer.
# - Overlay maps on the original aspect-preserved images.
# - Save one contact sheet per class with true label, prediction, confidence, and run ID.
#
# Expected output:
# - Traceable Grad-CAM sheets for correct and incorrect cases.
# - No causal claim based only on a heatmap.

> **Interpretation to write after running:** Describe recurring attention patterns and whether shortcut-like border/background focus appears.

### 12.3 Build the failure taxonomy

The rubric rewards honest analysis of limitations, not only successful examples.

In [ ]:
# TODO:
# - Review the deterministic cases and assign the fixed failure taxonomy.
# - Use ambiguity, weak data, shortcut, transform, imbalance, or model limitation.
# - Create a contact sheet with ID, labels, confidence, ArticleType, year group, and short note.
# - Save the taxonomy table and report-ready figure.
#
# Expected output:
# - A failure table with real IDs and balanced examples.
# - At least one honest limitation supported by evidence.

> **Interpretation to write after running:** Separate data limitations from model limitations and state which failure type matters most in real use.

## 13. Statistical and external comparison

Measure uncertainty between finalists and compare literature only when assumptions are made explicit.

### 13.1 Paired family bootstrap

Rows inside a product family may be related, so uncertainty should preserve those blocks.

In [ ]:
# TODO:
# - Pair finalist OOF predictions by product ID.
# - Bootstrap product_family_group blocks with the frozen seed and enough resamples.
# - Calculate the 95% interval for the macro-F1 difference and save the bootstrap distribution.
# - Mark the interval as conservative because family groups are not verified SKUs.
#
# Expected output:
# - A paired difference estimate, 95% interval, and distribution figure.
# - No independent-row bootstrap.

> **Interpretation to write after running:** State whether the evidence clearly separates finalists or supports the predeclared near-tie rule.

### 13.2 Qualified literature comparison

Published work uses different targets, splits, pretraining, resolution, or multimodal inputs, so scores are not direct benchmarks.

In [ ]:
# TODO:
# - Build a comparison table for the cited ResNet-BERT and Condition-CNN studies.
# - Record dataset, target, split, image resolution, pretraining, input modalities, and metric.
# - Mark score comparability as no unless every condition matches.
# - Save the table for the report appendix.
#
# Expected output:
# - A transparent assumptions table with authoritative citations.
# - No claim that unlike scores are directly equal.

> **Interpretation to write after running:** Explain what can be learned from the papers and why their numeric results cannot validate this exact Season task.

## 14. Ultimate Judgement and freeze

Apply the predeclared rule, reject alternatives explicitly, and freeze all choices before holdout.

### 14.1 Apply the final decision scorecard

The selected model must balance class quality, robustness, calibration, and deployment cost.

In [ ]:
# TODO:
# - Join leaderboard, shortcut, robustness, and calibration evidence by run ID.
# - Add bootstrap and efficiency evidence.
# - Apply the frozen ranking and near-tie rule without changing weights after viewing results.
# - Record the winner, every rejected alternative, evidence, cost, and limitation.
# - Save results/evidence/task2/ultimate_judgement.csv.
#
# Expected output:
# - One complete scorecard and one eligible scratch-trained winner.
# - Explicit reasons for rejecting every finalist and improvement.

> **Interpretation to write after running:** Write the final evidence-backed judgement in plain language, including the strongest limitation and the practical trade-off.

### 14.2 Write the freeze manifest

The manifest proves that model choices were locked before independent evaluation.

In [ ]:
# TODO:
# - Freeze run ID, config hash, split digest, labels, transform, and loss.
# - Freeze seed, metric, epoch, checkpoint, and calibration rules.
# - Record the decision timestamp and hash the manifest.
# - Save results/evidence/task2/freeze_manifest.json and display it.
#
# Expected output:
# - A complete, hashed, immutable freeze record.
# - No holdout metric or holdout-derived choice.

> **Interpretation to write after running:** Confirm that Notebook 06 can evaluate this exact decision without any remaining modelling choice.

### 14.3 Refit the frozen model on all development data

Final refit uses all labelled development rows only after CV selection is complete.

In [ ]:
# TODO:
# - Refit fold-learned preprocessing values and class weights on all valid development rows.
# - Train the frozen scratch configuration for the median best epoch from CV.
# - Apply development-OOF temperature scaling if it was selected.
# - Save models/task2_season.pt plus config, label map, and hashes.
#
# Expected output:
# - One final scratch-trained checkpoint and inference bundle.
# - Hashes matching the freeze manifest and no holdout-based early stopping.

> **Interpretation to write after running:** State the exact refit rule and verify that nothing was changed after the freeze.

## 15. Handoff to final evaluation

Audit every artifact and hand the frozen Task 2 component to Notebook 06 without opening holdout here.

### 15.1 Audit final artifacts and notebook evidence

Artifact traceability and clean notebook export are separate final checks.

#### 15.1.1 Artifact and traceability audit

Every final claim must map back to a run, configuration, prediction file, table, figure, or checkpoint.

In [ ]:
# TODO:
# - Check registry completeness, OOF coverage, and figure/table paths.
# - Check checkpoint hashes and inference smoke tests.
# - Verify that report numbers are generated rather than hand-copied.
# - Display a pass/fail definition-of-done checklist.
#
# Expected output:
# - All Task 2 artifacts exist and hashes match.
# - No missing run-to-claim link.

> **Interpretation to write after running:** List any unresolved artifact. The handoff remains blocked until all required evidence passes.

#### 15.1.2 Clean Run All and HTML export

A fresh execution proves that Notebook 03 is reproducible and presentation-ready.

In [ ]:
# TODO:
# - Restart the kernel and run all cells in order in the locked environment.
# - Check for stale outputs, hidden state, errors, and manual result edits.
# - Export results/notebooks/03_task2_season.html and record its digest.
#
# Expected output:
# - A clean successful Run All.
# - A fresh HTML export with a traceable digest.

> **Interpretation to write after running:** Confirm that the notebook is self-contained as a narrative and that every displayed result was regenerated.

### 15.2 Verify the sealed-holdout handoff

Notebook 03 prepares the final model but must never open protected evaluation labels.

In [ ]:
# TODO:
# - Assert that this notebook used only redacted loaders and development OOF evidence.
# - Package the frozen checkpoint, manifest, inference function, and allowed labels.
# - Include unresolved risks for Notebook 06.
# - Display the handoff summary with status READY only when every audit passes.
#
# Expected output:
# - A complete Notebook 06 handoff with holdout still sealed.
# - No holdout metric, prediction, or model change in this notebook.

> **Interpretation to write after running:** State whether the handoff is ready and name the remaining risks that Notebook 06 must report after its single evaluation.